In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tensorflow==2.10
!pip install keras
!pip install opencv-python
!pip install scikit-image
!pip install pycocotools
!pip install git+https://github.com/matterport/Mask_RCNN.git

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os
import random
from tensorflow import keras

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F
from pycocotools import mask as mask_utils
from PIL import Image

import os
import sys
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt

from mrcnn import model as modellib
from skimage import draw

# Import Mask RCNN
from mrcnn.config import Config
from mrcnn import utils  # Correct import for utils
from mrcnn.visualize import display_instances



In [ ]:
ROOT_DIR = os.path.abspath(".")

# Define the configuration for training
class BrainTumorConfig(Config):
    NAME = "brain_tumor"

    IMAGES_PER_GPU = 2
    NUM_CLASSES = 1 + 1  # Background + brain tumor class
    STEPS_PER_EPOCH = 100
    DETECTION_MIN_CONFIDENCE = 0.9

config = BrainTumorConfig()

# Custom dataset class
class BrainTumorDataset(utils.Dataset):
    def load_brain_tumor(self, dataset_dir, subset):
        """Load a subset of the brain tumor dataset.
        dataset_dir: Root directory of the dataset.
        subset: Subset to load (train or val)
        """
        self.add_class("brain_tumor", 1, "tumor")

        # Load annotations from JSON files
        annotations_file = os.path.join(dataset_dir, f"annotation_{subset}.json")
        with open(annotations_file, 'r') as f:
            annotations = json.load(f)

        # Add images
        for annotation in annotations['images']:
            image_path = os.path.join(dataset_dir, annotation['file_name'])
            image_id = annotation['id']
            width = annotation['width']
            height = annotation['height']

            self.add_image(
                "brain_tumor",
                image_id=image_id,
                path=image_path,
                width=width,
                height=height,
                annotations=annotations['annotations']
            )

    def load_mask(self, image_id):
        """Generate instance masks for an image."""
        info = self.image_info[image_id]
        annotations = [a for a in info['annotations'] if a['image_id'] == image_id]

        masks = np.zeros([info['height'], info['width'], len(annotations)], dtype=np.uint8)
        class_ids = np.array([self.map_source_class_id("brain_tumor.tumor")] * len(annotations))

        for i, annotation in enumerate(annotations):
            polygons = annotation['segmentation']
            mask = np.zeros((info['height'], info['width']), dtype=np.uint8)
            rr, cc = skimage.draw.polygon(polygons[1::2], polygons[0::2])
            mask[rr, cc] = 1
            masks[:, :, i] = mask

        return masks, class_ids


# Load datasets
dataset_train = BrainTumorDataset()
dataset_train.load_brain_tumor('/content/drive/MyDrive/dataset files/brain tumor-kaggle/Br35H-Mask-RCNN', 'train')
dataset_train.prepare()

dataset_test = BrainTumorDataset()
dataset_test.load_brain_tumor('/content/drive/MyDrive/dataset files/brain tumor-kaggle/Br35H-Mask-RCNN', 'test')
dataset_test.prepare()


In [ ]:
!wget https://github.com/matterport/Mask_RCNN/releases/download/v2.0/mask_rcnn_coco.h5

In [ ]:
model = modellib.MaskRCNN(mode="training", config=config, model_dir='/content/logs')

# Load a pre-trained model or start from scratch
model.load_weights('mask_rcnn_coco.h5', by_name=True, exclude=["mrcnn_class_logits", "mrcnn_bbox_fc", "mrcnn_bbox", "mrcnn_mask"])


In [ ]:
model.train(dataset_train, dataset_test,
            learning_rate=config.LEARNING_RATE,
            epochs=30,
            layers='heads')


In [ ]:
# Switch to inference mode
model = modellib.MaskRCNN(mode="inference", config=config, model_dir='/content/logs')

# Load trained weights
model.load_weights('/content/logs/mask_rcnn_brain_tumor.h5', by_name=True)

# Test on a new image
image_path = '/content/brain-tumor-dataset/test/image.jpg'
image = cv2.imread(image_path)

# Detect
results = model.detect([image], verbose=1)

# Visualize results
r = results[0]
display_instances(image, r['rois'], r['masks'], r['class_ids'],
                  ['BG', 'tumor'], r['scores'])
